In [28]:
import sklearn.cluster as sk
import itertools
import sys
import matplotlib.pyplot as plt
#from kneed import KneeLocator
from sklearn.metrics import silhouette_score
from pathlib import Path
import geopandas as gpd
path = Path().cwd().parent / "src" #to add the src directory to the path regognized by Python
sys.path.append(str(path))
import methods_two_point_correlation as mtpc
import save_load_pickle as slp
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.optimize import curve_fit
# Bookkeeping of important path in the project
data_dir = Path().cwd().parent/Path("data")
data_city = data_dir / "cities"
data_map = data_dir / "map"
out_dir = Path().cwd().parent/Path("out")
plot_dir = out_dir / Path("plot")

In [29]:
name = "Belgium" #name of the system considered
name_random = f"Random{name}" #How the name of the random catalo will be saved
threshold = 1 # minimal number of inhabitants for the city to be considered
N_run = 5 # number of random catalog generated for computing the 2PCF
size = 7000 #number of points each random catalog has
k = 3  #number of time the 2PCF computing process is repeated to estimate variance
rmax = 3e5 #maximal distance between two points considered, if None, taken the max from data
scale = "log" #scale at which to bin the data, recommended "log"
nbins = 20 # number of bins to consider in the computation of 2PCF (betweem rmin and rmax)

In [30]:
##Preprocessing
crs = mtpc.crs_selector(name) # Coordinate reference system of the country considered
path_city = data_city / f"{name}_cities.csv" #Path of the cities 
path_border = data_map / f"{name}.geojson" # Path of the multipolygon of the territory of the country
gdf_city = mtpc.load_df_to_gdf(path_city,threshold) # Geopandas of the dataset
gdf_edge = gpd.read_file(path_border) #Geopandas of the polygon border of the dataset
gdf_projected = gdf_city.to_crs(crs) # Projection in the right coordinate system
coord = gdf_projected.get_coordinates() # Coordinates of the points in dataset

In [31]:
gdf_projected

,Unnamed: 0,city,city_ascii,city_alt,lat,lng,country,iso2,iso3,admin_name,admin_name_ascii,admin_code,admin_type,capital,density,population,population_proper,ranking,timezone,same_name,id,geometry
0,1769739,Brussels,Brussels,Bruxelles|Brussel|Brüssel,50.8467,4.3525,Belgium,BE,BEL,Brussels-Capital Region,Brussels-Capital Region,BE-BRU,region,primary,7465.2,1235192.0,1235192.0,1,Europe/Brussels,False,1056469830,POINT (648855.036 670699.945)
1,1769740,Antwerp,Antwerp,Anvers|Antwerpen,51.2178,4.4003,Belgium,BE,BEL,Flanders,Flanders,BE-VLG,region,minor,2600.0,536079.0,536079.0,2,Europe/Brussels,False,1056168623,POINT (652198.521 711984.241)
2,1769741,Gent,Gent,Gand|Ghent,51.0536,3.7253,Belgium,BE,BEL,Flanders,Flanders,BE-VLG,region,minor,1680.2,265086.0,265086.0,2,Europe/Brussels,False,1056062897,POINT (604881.346 693905.889)
3,1769742,Charleroi,Charleroi,NaN,50.4000,4.4333,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,minor,1977.0,201816.0,201816.0,2,Europe/Brussels,False,1056263311,POINT (654594.973 621014.289)
4,1769743,Liège,Liege,Lige|Lüttich|Leodicum|Luik,50.6397,5.5706,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,minor,2844.5,195278.0,195278.0,2,Europe/Brussels,False,1056513284,POINT (735011.912 648372.85)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815,1770554,Chaumont,Chaumont,NaN,49.9205,5.6722,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,NaN,46.3,34.0,34.0,4,Europe/Brussels,True,1056855567,POINT (743612.065 568511.979)
816,1770555,Belle Eau,Belle Eau,NaN,49.9519,5.6161,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,NaN,46.3,32.0,32.0,4,Europe/Brussels,False,1056145310,POINT (739524.858 571934.252)
817,1770556,Oberhausen,Oberhausen,NaN,50.1572,6.1399,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,NaN,39.5,27.0,27.0,4,Europe/Brussels,False,1056467055,POINT (776560.225 595531.238)
818,1770557,Rhisnes,Rhisnes,NaN,50.4911,4.7981,Belgium,BE,BEL,Wallonia,Wallonia,BE-WAL,region,NaN,35.8,24.0,24.0,3,Europe/Brussels,False,1056705298,POINT (680470.155 631236.77)


In [32]:
def compute_DD_SP(gdf_projected):
    coord_data = gdf_projected.get_coordinates().to_numpy()
    DD = distance.pdist(coord_data).astype(np.float32)#np.triu(distance.cdist(coord_data,coord_data)).astype(np.int32)
    #DD = np.ravel(DD)
    #DD = DD[DD != 0]
    return DD

def compute_one_DR_RR_SP(gdf_projected,gdf_edge, size, crs):
    coord_random = generate_random_point(gdf_edge, size, crs)#generate_sobol_point(gdf_edge, size, crs)#generate_random_point(gdf_edge, size, crs)
    coord_data = gdf_projected.get_coordinates().to_numpy()
    DR = distance.cdist(coord_data,coord_random).astype(np.float32)
    DR = np.ravel(DR)
    DR = DR[DR != 0]
    RR = distance.pdist(coord_random).astype(np.float32)#np.triu(distance.cdist(coord_random,coord_random)).astype(np.int32)
    #RR = np.ravel(RR)
    #RR = RR[RR != 0]
    return DR,RR#,len(coord_random)

def compute_two_point_correlation_2019_metadata(gdf_projected,gdf_edge,crs,N_run,Nr_prime,rmin,Nd,rmax,scale,nbins=20):
    Nr = Nr_prime*N_run
    #Nr = len(gdf_projected)
    metadata_min = np.min(population)
    metadata_max = np.max(population)
    meta_edges = np.logspace(np.log10(metadata_min), np.log10(metadata_max), nbins)
    for h in range(len(meta_edges)-1):
        
    DD = compute_DD_SP(gdf_projected)
    DR,RR = compute_one_DR_RR_SP(gdf_projected,gdf_edge, Nr_prime, crs)
    for i in range(N_run-1):
        print(i)
        DR_i,RR_i = compute_one_DR_RR_SP(gdf_projected,gdf_edge, Nr_prime, crs)
        RR = np.concatenate((RR,RR_i))
        DR = np.concatenate((DR,DR_i))
    if not rmin:
        rmin = compute_rmin(gdf_projected)
    if rmax:
        rmax = rmax
    else:
        rmax = np.max(DD)+1#find_rmax_meaningful(gdf_edge,crs,100000)
    if scale =="log":
        r_edges = np.logspace(np.log10(rmin), np.log10(rmax), nbins)
    elif scale == "lin":
        r_edges = np.linspace(rmin,rmax,nbins)
    hist_DD = binning_data(DD,nbins,r_edges)
    print("max DD", np.max(DD))
    print("min DD",np.min(DD))
    #hist_DD = hist_DD[0:(len(hist_DD)-1)]
    del DD
    hist_DR = binning_data(DR,nbins,r_edges)
    del DR
    hist_RR = binning_data(RR,nbins,r_edges)
    print("maxRR",np.max(RR))
    print("minRR",np.min(RR))
    del RR
    print(rmin,rmax)
    if len(hist_RR) == len(hist_DD)+1:
        hist_RR = hist_RR[0:(len(hist_RR)-1)]
    if len(hist_DR) == len(hist_DD)+1:
        hist_DR = hist_DR[0:(len(hist_DR)-1)]
    print(np.sum(hist_DD)*(2/(Nd*(Nd-1))))
    print(np.sum(hist_DR)/((Nd*(Nr))))
    print(np.sum(hist_RR)*(2/(Nr*(Nr_prime-1))))
    xi= compute_LS_correlation_2019(hist_DD,hist_DR,hist_RR,Nd,Nr_prime,Nr)
    return r_edges,xi

IndentationError: expected an indented block after 'for' statement on line 25 (2914654203.py, line 27)

In [33]:
metadata_name = "population"
population = gdf_projected[metadata_name].values

In [168]:
from scipy.spatial import distance
def compute_DD_meta(gdf_projected_meta,gdf_projected_meta_prime):
    coord_meta = gdf_projected_meta.get_coordinates().to_numpy()
    coord_meta_prime = gdf_projected_meta_prime.get_coordinates().to_numpy()
    DD = np.triu(distance.cdist(coord_meta,coord_meta_prime)).astype(np.int32)
    DD = np.ravel(DD)
    DD = DD[DD != 0]
    return DD

def compute_DR_meta(gdf_projected_meta,gpd_random_with_meta_prime,gpd_random_with_meta):
    coord_meta = gdf_projected_meta.get_coordinates().to_numpy()
    coord_meta_random_prime = gpd_random_with_meta_prime[0].get_coordinates().to_numpy()
    coord_meta_random = gpd_random_with_meta[0].get_coordinates().to_numpy()
    #DR = np.triu(distance.cdist(coord_meta,coord_meta_random_prime)).astype(np.int32)
    DR = distance.cdist(coord_meta,coord_meta_random_prime).astype(np.int32)
    DR = np.ravel(DR)
    DR = DR[DR != 0]
    RR = np.triu(distance.cdist(coord_meta_random,coord_meta_random_prime)).astype(np.int32)
    RR = np.ravel(RR)
    RR = RR[RR != 0]
    return DR,RR
    
# NEED TO ADAPT TO ANY NUMBER OF POINTS 
def generate_random_geopandas_with_pop(gdf_edge,gdf_projected,crs):
    N = len(gdf_projected)
    sample = gdf_edge.sample_points(N)
    gdf_random = sample.to_crs(crs)
    sample = sample.explode(ignore_index=True)
    gpd_rand_pop = try_panda = pd.concat([sample,gdf_projected[metadata_name]],keys=["geometry", "population"],axis=1,ignore_index=True)
    gpd_rand_pop[0] = gpd_rand_pop[0].to_crs(crs)
    return gpd_rand_pop

In [176]:
nbins = 5
metadata_min = np.min(population)
metadata_max = np.max(population)+1
meta_edges = np.logspace(np.log10(metadata_min), np.log10(metadata_max), nbins)
gpd_random_with_pop = generate_random_geopandas_with_pop(gdf_edge,gdf_projected,crs)
DD,DR,RR = np.zeros([nbins-1,nbins-1],dtype=list),np.zeros([nbins-1,nbins-1],dtype=list),np.zeros([nbins-1,nbins-1],dtype=list)
for h in range(len(meta_edges)-1):
    gdf_projected_low = gdf_projected[gdf_projected[metadata_name] >= meta_edges[h]]
    gdf_projected_meta = gdf_projected_low[gdf_projected_low[metadata_name] < meta_edges[h+1]]
    gpd_random_with_pop_low = gpd_random_with_pop[gpd_random_with_pop[1] >= meta_edges[h]]
    gpd_random_with_meta = gpd_random_with_pop_low[gpd_random_with_pop_low[1] < meta_edges[h+1]]
    print(len(gdf_projected_meta),meta_edges[h+1])
    for h_prime in range(len(meta_edges)-1):
        gdf_projected_low = gdf_projected[gdf_projected[metadata_name] >= meta_edges[h_prime]]
        gdf_projected_meta_prime = gdf_projected_low[gdf_projected_low[metadata_name] < meta_edges[h_prime+1]]
        DD[h][h_prime]=compute_DD_meta(gdf_projected_meta,gdf_projected_meta_prime)
        #DD.append(compute_DD_meta(gdf_projected_meta,gdf_projected_meta_prime))
        gpd_random_with_pop_low = gpd_random_with_pop[gpd_random_with_pop[1] >= meta_edges[h_prime]]
        gpd_random_with_meta_prime = gpd_random_with_pop_low[gpd_random_with_pop_low[1] < meta_edges[h_prime+1]]
        DR_i,RR_i = compute_DR_meta(gdf_projected_meta,gpd_random_with_meta_prime,gpd_random_with_meta)
        DR[h][h_prime]= DR_i
        RR[h][h_prime]= RR_i

rmin = None
if not rmin:
    rmin = mtpc.compute_rmin(gdf_projected)
if rmax:
    rmax = rmax
else:
    rmax = np.max(DD)+1#find_rmax_meaningful(gdf_edge,crs,100000)
scale="log"
if scale =="log":
    r_edges = np.logspace(np.log10(rmin), np.log10(rmax), nbins)
elif scale == "lin":
    r_edges = np.linspace(rmin,rmax,nbins)
hist_DD = np.zeros([nbins-1,nbins-1],dtype=list)
hist_DR = np.zeros([nbins-1,nbins-1],dtype=list)
hist_RR = np.zeros([nbins-1,nbins-1],dtype=list)
xi = np.zeros([nbins-1,nbins-1],dtype=list)

sum_DD,sum_DR,sum_RR = 0,0,0
Nd=820
Nr=820
Nr_prime = 820
for i in range(len(DD)):
    for j in range(len(DD)):
        hist_DD[i][j] = mtpc.binning_data(DD[i][j],nbins,r_edges)
        hist_DR[i][j] = mtpc.binning_data(DR[i][j],nbins,r_edges)
        hist_RR[i][j] = mtpc.binning_data(RR[i][j],nbins,r_edges)
        print(RR[i][j])
        sum_DD += np.sum(hist_DD[i][j])*(2/(Nd*(Nd-1)))
        sum_DR += np.sum(hist_DR[i][j])/((Nd*(Nr)))
        sum_RR += np.sum(hist_RR[i][j])*(2/(Nr*(Nr_prime)))
        print(hist_DD[i][j])
        print(hist_DR[i][j])
        print(hist_RR[i][j])
        xi[i][j] = mtpc.compute_LS_correlation_2019(DD[i][j],DR[i][j],RR[i][j],Nd,Nr_prime,Nr)

33 303.38835428882345
245 4844.447027267405
524 77355.20058116008
18 1235193.0000000002
[44072  1686  2713  2512  8126  3218 28599 28526 45554 10903 26327  7076
 22958  4104  5804 42501 31590 26652 38044 15662 15028 38708 33516 23346
 30312 35273 35682 35133 33068 33533 32216 31808 42405 46678 46400 36038
 47044 15566 15678  2248 33575 18079 37797 21717 43066 40073  4656 13490
 18520  8681 30821 32519 10532 15365 26372 20220 16381 18064 19294 25850
 26031 27471 29490  4299  4049  6440  4734 26922 26847 43879  9224 24643
  5483 21272  3709  4529 40817 29904 24966 36357 14065 13511 37026 31847
 21778 28685 33623 34054 33520 31576 32046 30765 30414   342 10656   611
 31172 31091 48131 13317 28854  9274 25429  4926  7528 45021 34078 29093
 40512 17746 16912 41097 35814 25304 32437 37481 37804 37203 34794 35244
 33845 33303 10369   706 30887 30804 47846 13012 28560  8954 25127  4586
  7191 44725 33778 28787 40209 17415 16574 40784 35493 24966 32106 37154
 37472 36868 34452 34902 33503 32961

ValueError: operands could not be broadcast together with shapes (1089,) (528,) 

In [170]:

sum_DR

0.9999999999999998

In [138]:
np.sort(RR[1][0])

array([56254, 60169, 61790, 65684, 67459, 69067, 69764, 70417, 70682,
       71225, 74564, 74618, 75085, 76433, 77123], dtype=int32)

In [137]:
np.sort(DD[0][1])

array([  1874,   3558,   3760,   4137,   5365,   5711,   6849,   8144,
         9231,   9446,  40724,  43504,  47977,  79835,  86302,  87293,
       158782, 159661, 162948, 164444], dtype=int32)

In [92]:
lol = generate_random_geopandas_with_pop(gdf_edge,gdf_projected)
coord_random = lol[0].get_coordinates().to_numpy()

In [93]:
coord_random

array([[ 2.57408352, 51.0326741 ],
       [ 2.65984134, 51.07053468],
       [ 2.67867107, 50.86369307],
       ...,
       [ 6.24573193, 50.29713151],
       [ 6.3114758 , 50.47853808],
       [ 6.35786226, 50.36764523]])

In [79]:
sample = gdf_edge.sample_points(820)
gdf_random = sample.to_crs(crs).explode()

In [80]:
sample = sample.explode(ignore_index=True)

In [81]:
try_panda = pd.concat([sample,gdf_projected[metadata_name]],keys=["geometry", "population"],axis=1,ignore_index=True)

In [82]:
try_panda

,0,1
0,POINT (2.58401 51.08554),1235192.0
1,POINT (2.6405 50.88846),536079.0
2,POINT (2.66137 51.01843),265086.0
3,POINT (2.67642 51.1112),201816.0
4,POINT (2.67687 50.98985),195278.0
...,...,...
815,POINT (6.24108 50.48219),34.0
816,POINT (6.26257 50.2985),32.0
817,POINT (6.26605 50.40077),27.0
818,POINT (6.27893 50.4317),24.0


In [19]:
def compute_DD_meta(gdf_projected,meta_edges):
    coord_data = gdf_projected.get_coordinates().to_numpy()
    DD = distance.pdist(coord_data).astype(np.float32)#np.triu(distance.cdist(coord_data,coord_data)).astype(np.int32)
    #DD = np.ravel(DD)
    #DD = DD[DD != 0]
    return DD

array([1.90000000e+01, 3.40461794e+01, 6.10074912e+01, 1.09319579e+02,
       1.95890210e+02, 3.51016486e+02, 6.28987908e+02, 1.12708606e+03,
       2.01963023e+03, 3.61898384e+03, 6.48487228e+03, 1.16202697e+04,
       2.08224099e+04, 3.73117634e+04, 6.68591048e+04, 1.19805109e+05,
       2.14679276e+05, 3.84684692e+05, 6.89318108e+05, 1.23519200e+06])

In [14]:
metadata_max

1235192.0